## Double Pendulum Phase Space

Phase space is obtained by plotting graph of $\dot{p}_i$ against $p_i$.

For our case, we have two phase space since we have two Cartesian coordinates $\{x,y\}$. Therefore, we must plot $\{\dot{x},x\}$ and $\{\dot{y},y\}.$

---
Cartesian coordinates (pendulum 1) :
$$
\begin{cases}
x_1=l_1\sin\theta_1\\
y_1=-l_1\cos\theta_1
\end{cases}
$$
Cartesian coordinates (pendulum 2) :
$$
\begin{cases}
x_2=l_1\sin\theta_1+l_2\sin\theta_2\\
y_2=-(l_1\cos\theta_1+l_2\cos\theta_2)
\end{cases}
$$
Velocities (pendulum 1) :
$$
\begin{cases}
\dot{x}_1=l_1\dot{\theta}_1\cos\theta_1\\
\dot{y}_1=l_1\dot{\theta}_1\sin\theta_1
\end{cases}
$$
Velocities (pendulum 2) :
$$
\begin{cases}
\dot{x}_2=l_1\dot{\theta}_1\cos\theta_1+l_2\dot{\theta}_2\cos\theta_2\\
\dot{y}_2=l_1\dot{\theta}_1\sin\theta_1+l_2\dot{\theta}_2\sin\theta_2
\end{cases}
$$

# Helper Function

In [ ]:
import numpy as np
from scipy.linalg import solve

def position_bob1(θ1, l1):
    """Returns the (x, y) position of the first bob."""
    x1 = l1 * np.sin(θ1)
    y1 = -l1 * np.cos(θ1)
    return x1, y1

def velocity_bob1(θ1, ω1, l1):
    """Returns the (vx, vy) velocity of the first bob."""
    vx1 = l1 * ω1 * np.cos(θ1)
    vy1 = l1 * ω1 * np.sin(θ1)
    return vx1, vy1

def position_bob2(x1, y1, θ2, l2):
    """Returns the (x, y) position of the second bob."""
    x2 = x1 + l2 * np.sin(θ2)
    y2 = y1 - l2 * np.cos(θ2)
    return x2, y2

def velocity_bob2(vx1, vy1, θ2, ω2, l2):
    """Returns the (vx, vy) velocity of the second bob."""
    vx2 = vx1 + l2 * ω2 * np.cos(θ2)
    vy2 = vy1 + l2 * ω2 * np.sin(θ2)
    return vx2, vy2

def double_pendulum_derivatives(t, y, m1, m2, l1, l2, g):
    """
    Returns the derivatives of the double pendulum system.
    """
    # Parameters
    θ1, θ2, ω1, ω2 = y # state vector
    delta = θ2 - θ1
    m12 = m1 + m2
    
    # Defining the vectors as matrix functions
    A = np.array([
        [m12 * l1, m2 * l2 * np.cos(delta)],
        [l1 * np.cos(delta), l2]
    ])
    
    b = np.array([
        m2 * l2 * abs(ω2**2) * np.sin(delta) - m12 * g * np.sin(θ1),
        -l1 * abs(ω1**2) * np.sin(delta) - g * np.sin(θ2)
    ])
    
    # Solve for acceleration
    α1, α2 = solve(A, b)
    
    return np.array([ω1, ω2, α1, α2])

def rk4_step(f, t, y, dt, *args):
    """
    4th Order Runge-Kutta step
    """
    k1 = f(t, y, *args)
    k2 = f(t + dt/2, y + dt/2 * k1, *args)
    k3 = f(t + dt/2, y + dt/2 * k2, *args)
    k4 = f(t + dt, y + dt * k3, *args)

    return y + dt/6 * (k1 + 2*k2 + 2*k3 + k4)

In [ ]:
def simulate_double_pendulum(θ1, θ2, ω1=0, ω2=0, 
                             t_max=10, dt=0.01, 
                             m1=1, m2=1, 
                             l1=1, l2=1, 
                             g=9.81):
    """
    Simulate the double pendulum
    """
    # Initial state
    y = np.array([θ1, θ2, ω1, ω2])
    
    # Time array
    t_values = np.arange(0, t_max, dt)
    
    # Store results
    results = np.zeros((len(t_values), 4))
    results[0] = y
    
    # Simulation loop
    for i in range(1, len(t_values)):
        # print(i)
        results[i] = rk4_step(double_pendulum_derivatives, t_values[i-1], 
                             results[i-1], dt, m1, m2, l1, l2, g)

    return t_values, results

# Results

In [ ]:
# Cartesian coordinates of bobs
dt = 0.01 
l1 = l2 = m1 = m2 = 1.0

# Obtain simulation results
t, results = simulate_double_pendulum(
    θ1=90*np.pi/180, # 90 degrees
    θ2=45*np.pi/180, # 45 degrees  
    ω1=0,            # starting from rest
    ω2=0,            # starting from rest
    t_max=10,        # simulate for 10 seconds
    dt=0.01          # time step of 0.01 seconds
)

# Extract results
theta1, theta2, omega1, omega2 = results.T
x1, y1 = position_bob1(theta1, l1)
x2, y2 = position_bob2(x1, y1, theta2, l2)
v_x1, v_y1 = velocity_bob1(theta1, omega1, l1)
v_x2, v_y2 = velocity_bob2(v_x1, v_y1, theta2, omega2, l2)

# Animation

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from IPython.display import HTML

# Plot style
plt.style.use('seaborn-v0_8-darkgrid')
plt.rcParams['font.family'] = "Liberation Serif"
plt.rcParams['figure.dpi'] = 100

# Create plots
fig, (ax,ax_top,ax_bottom) = plt.subplots(1, 3, figsize=(12, 4))

###############################
# Configuration Space
###############################

# Customize axes
ax.set_title(r"Double Pendulum Simulation")
ax.set_xlabel('X-axis')
ax.set_ylabel('Y-axis')
ax.set_xlim(-2, 2)
ax.set_ylim(-2, 2)
ax.axhline(y=0, ls=':', lw=0.5, color='black')
ax.axvline(x=0, ls=':', lw=0.5, color='black')

# Pendulum 1
line5, = ax.plot([], [], c='darkgreen', lw=0.5, ls='--')
line7, = ax.plot([], [], c='red', lw=0.5, ls='--')
scat5 = ax.scatter([], [], c='r', s=10, label=f'Pendulum 1')

# Pendulum 2
line6, = ax.plot([], [], c='purple', lw=0.5, ls='--')
line8, = ax.plot([], [], c='blue', lw=0.5, ls='--')
scat6 = ax.scatter([], [], c='b', s=10, label=f'Pendulum 2')

# Legends
ax.legend(frameon=True)

###############################
# Phase Space (x)
###############################

# Customize axes
ax_top.set_title(r"Phase Space $(x)$")
ax_top.set_xlabel(r'Position, $x$')
ax_top.set_ylabel(r'Velocity, $\dot{x}$')
ax_top.set_xlim(-5, 5)
ax_top.set_ylim(-5, 5)
ax_top.axhline(y=0, ls=':', lw=0.5, color='black')
ax_top.axvline(x=0, ls=':', lw=0.5, color='black')

# Phase Space (x)
# Pendulum 1
line1, = ax_top.plot([], [], c='red', lw=0.5, ls='--')
scat1 = ax_top.scatter([], [], c='r', s=10, label=r'$\dot{x}_1$ against $x_1$')
# Pendulum 2
line2, = ax_top.plot([], [], c='blue', lw=0.5, ls='--')
scat2 = ax_top.scatter([], [], c='b', s=10, label=r'$\dot{x}_2$ against $x_2$')

# Legends
ax_top.legend(fontsize='xx-small', frameon=True)

###############################
# Phase Space (y)
###############################

# Customize axes
ax_bottom.set_title(r"Phase Space $(y)$")
ax_bottom.set_xlabel(r'Position, $y$')
ax_bottom.set_ylabel(r'Velocity, $\dot{y}$')
ax_bottom.set_xlim(-5, 5)
ax_bottom.set_ylim(-5, 5)
ax_bottom.axhline(y=0, ls=':', lw=0.5, color='black')
ax_bottom.axvline(x=0, ls=':', lw=0.5, color='black')

# Phase Space (y)
# Pendulum 1
line3, = ax_bottom.plot([], [], c='red', lw=0.5, ls='--')
scat3 = ax_bottom.scatter([], [], c='r', s=10, label=r'$\dot{y}_1$ against $y_1$')
# Pendulum 2
line4, = ax_bottom.plot([], [], c='blue', lw=0.5, ls='--')
scat4 = ax_bottom.scatter([], [], c='b', s=10, label=r'$\dot{y}_2$ against $y_2$')

# Legends
ax_bottom.legend(fontsize='xx-small', frameon=True)

###############################
# Animation
###############################

def update(frame):
    # For each frame, update the data stored on each artist

    # Configuration Space ------------------
    # Line (string)
    line5.set_data([0, x1[frame]], [0, y1[frame]])
    line6.set_data([x1[frame], x2[frame]], [y1[frame], y2[frame]])

    # Line (tracing)
    line7.set_data(x1[:frame], y1[:frame])
    line8.set_data(x2[:frame], y2[:frame])
    
    # Scatter
    data5 = np.stack([x1[frame], y1[frame]]).T
    data6 = np.stack([x2[frame], y2[frame]]).T
    scat5.set_offsets(data5)
    scat6.set_offsets(data6)

    # Phase Space (x) ------------------
    # Line
    line1.set_data([x1[:frame]],[v_x1[:frame]])
    line2.set_data([x2[:frame]],[v_x2[:frame]])
    
    # Scatter
    data1 = np.stack([x1[frame], v_x1[frame]]).T
    data2 = np.stack([x2[frame], v_x2[frame]]).T
    scat1.set_offsets(data1)
    scat2.set_offsets(data2)

    # Phase Space (y) ------------------
    # Line
    line3.set_data([y1[:frame]],[v_y1[:frame]])
    line4.set_data([y2[:frame]],[v_y2[:frame]])
    
    # Scatter
    data3 = np.stack([y1[frame], v_y1[frame]]).T
    data4 = np.stack([y2[frame], v_y2[frame]]).T
    scat3.set_offsets(data3)
    scat4.set_offsets(data4)

    return line1,line2,line3,line4,line5,line6,line7,line8,scat1,scat2,scat3,scat4,scat5,scat6

ani = animation.FuncAnimation(fig=fig, func=update, frames=len(x1), interval=dt*1000, blit=True)

# Play the animation
# ani.save('phase-space-massless.gif', writer=animation.PillowWriter(fps=1/dt), dpi=200)
plt.close() # Hide extra image
HTML(ani.to_jshtml()) # Display inline
